In [2]:
import pandas as pd
import numpy as np

df_inv = pd.read_csv("../data/processed/08_investor_transactions_clean.csv")

df_inv.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [4]:
df_inv["transaction_date"] = pd.to_datetime(df_inv["transaction_date"])

In [5]:
df_inv["first_transaction_date"] = (
    df_inv.groupby("investor_id")["transaction_date"]
    .transform("min")
)

df_inv["cohort_year"] = df_inv["first_transaction_date"].dt.year

In [6]:
df_inv[[
    "investor_id",
    "transaction_date",
    "first_transaction_date",
    "cohort_year"
]].head()

,investor_id,transaction_date,first_transaction_date,cohort_year
0,INV003054,2024-01-01,2024-01-01,2024
1,INV002952,2024-01-01,2024-01-01,2024
2,INV003420,2024-01-01,2024-01-01,2024
3,INV003436,2024-01-01,2024-01-01,2024
4,INV004691,2024-01-01,2024-01-01,2024


In [7]:
sip_data = df_inv[
    df_inv["transaction_type"] == "SIP"
].copy()

avg_sip = (
    sip_data.groupby("cohort_year")["amount_inr"]
    .mean()
    .reset_index(name="avg_sip_amount")
)

In [8]:
investment_data = df_inv[
    df_inv["transaction_type"].isin(["SIP", "Lumpsum"])
].copy()

total_invested = (
    investment_data.groupby("cohort_year")["amount_inr"]
    .sum()
    .reset_index(name="total_invested")
)

In [9]:
fund_preference = (
    investment_data.groupby(
        ["cohort_year", "amfi_code"]
    )["amount_inr"]
    .sum()
    .reset_index()
)

In [10]:
top_fund = (
    fund_preference.loc[
        fund_preference.groupby("cohort_year")["amount_inr"].idxmax()
    ]
    .rename(columns={
        "amfi_code": "top_fund",
        "amount_inr": "top_fund_investment"
    })
)

In [11]:
cohort_analysis = (
    avg_sip
    .merge(total_invested, on="cohort_year", how="outer")
    .merge(
        top_fund[
            ["cohort_year", "top_fund", "top_fund_investment"]
        ],
        on="cohort_year",
        how="outer"
    )
    .sort_values("cohort_year")
)

cohort_analysis

,cohort_year,avg_sip_amount,total_invested,top_fund,top_fund_investment
0,2024,10996.885825,2258062304,119095,65025476
1,2025,13505.209581,18992635,119094,1283425


In [12]:
cohort_analysis.to_csv(
    "../data/processed/investor_cohort_analysis.csv",
    index=False
)